In [ ]:
import time
from pathlib import Path

import requests


In [20]:
API_KEY = "8rQcU4ZoNFmcjS24fCTmpUCCJp3G9uhr9fQ93lb2aQs"
API_URL = "https://www.datalab.to/api/v1/marker"

PDF_PATH = Path(
    r"D:\dack\research-agent\backend\crawled_papers\2023.findings-ijcnlp.15.pdf"
)
OUTPUT_DIR = Path("datalab_output")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)


In [ ]:
with open(PDF_PATH, "rb") as f:
    form_data = {
        "file": (PDF_PATH.name, f, "application/pdf"),
        "force_ocr": (None, "false"),
        "paginate": (None, "false"),
        "output_format": (None, "markdown"),
        "use_llm": (None, "false"),
        "strip_existing_ocr": (None, "false"),
        "disable_image_extraction": (None, "false"),
    }

headers = {"X-Api-Key": API_KEY}

response = requests.post(API_URL, headers=headers, files=form_data)

print("Status:", response.status_code)
print(response.text)


Status: 200
{"success":true,"error":null,"request_id":"cj5ReXuvIHUsIr5Q2eKFpw","request_check_url":"https://www.datalab.to/api/v1/marker/cj5ReXuvIHUsIr5Q2eKFpw","versions":null}


In [22]:
response.raise_for_status()
upload_result = response.json()

if not upload_result.get("success"):
    raise RuntimeError(upload_result.get("error", "Upload failed"))

request_check_url = upload_result.get("request_check_url")

if not request_check_url:
    raise RuntimeError("No request_check_url returned")

request_check_url


'https://www.datalab.to/api/v1/marker/cj5ReXuvIHUsIr5Q2eKFpw'

In [23]:
MAX_ATTEMPTS = 120
SLEEP_SECONDS = 5

result = None

for attempt in range(MAX_ATTEMPTS):
    r = requests.get(request_check_url, headers=headers)
    r.raise_for_status()

    result = r.json()
    status = result.get("status")

    print(f"Attempt {attempt + 1}: status = {status}")

    if status == "complete":
        print("✅ Conversion complete")
        break

    if status == "failed":
        raise RuntimeError(result.get("error", "Conversion failed"))

    time.sleep(SLEEP_SECONDS)
else:
    raise TimeoutError("⏰ Conversion timeout")


Attempt 1: status = processing
Attempt 2: status = processing
Attempt 3: status = processing
Attempt 4: status = processing
Attempt 5: status = processing
Attempt 6: status = complete
✅ Conversion complete


In [24]:
result.keys()


dict_keys(['status', 'output_format', 'chunks', 'json', 'markdown', 'html', 'extraction_schema_json', 'segmentation_results', 'images', 'metadata', 'success', 'error', 'parse_quality_score', 'page_count', 'total_cost', 'cost_breakdown', 'runtime', 'checkpoint_id', 'versions'])

In [25]:
markdown = result.get("markdown")

if not markdown:
    raise RuntimeError("No markdown returned")

md_path = OUTPUT_DIR / "output.md"
md_path.write_text(markdown, encoding="utf-8")

print("✅ Markdown saved to:", md_path)


✅ Markdown saved to: datalab_output\output.md


In [26]:
import base64

images = result.get("images", {})

if not images:
    print("No images returned")
else:
    for image_name, image_data in images.items():
        image_bytes = base64.b64decode(image_data)
        image_path = OUTPUT_DIR / "images" / image_name
        image_path.write_bytes(image_bytes)

        print("Saved image:", image_path)


Saved image: datalab_output\images\81bf7a5187d196cc72844283e3aeec49_img.jpg
Saved image: datalab_output\images\9ccd03fe518c562a3fe2d3119f50935e_img.jpg
Saved image: datalab_output\images\bec3b1d6f15b643228b0da0a7d47bdbd_img.jpg
